# 03 — Treinamento baseline YOLOv8n no Google Colab

Este notebook executa o **primeiro treinamento baseline** do projeto.

## Objetivo

Treinar um detector YOLOv8n utilizando o dataset HomeObjects-3K preparado anteriormente:

- `train`: treinamento do modelo;
- `val`: acompanhamento durante o treinamento;
- `test`: avaliação final independente.

O conjunto de **teste não é utilizado durante o treinamento**.

## Fluxo

```text
Dataset processado
       ↓
Google Drive
       ↓
Google Colab + GPU
       ↓
YOLOv8n pré-treinado
       ↓
Fine-tuning
       ↓
best.pt
       ↓
Avaliação no TEST
       ↓
Métricas + gráficos + pesos
```

---
**Antes de executar este notebook:**

No seu computador, compacte a pasta:

```text
data/processed/homeobjects_yolo
```

gerando:

```text
homeobjects_yolo.zip
```

O ZIP deve estar disponível no Google Drive em:

```text
Meu Drive/
└── TCC_Visao_Computacional/
    └── datasets/
        └── homeobjects_yolo.zip
```

## 1. Ativar GPU no Colab

No Google Colab, acesse:

```text
Ambiente de execução
    → Alterar tipo de ambiente de execução
    → Acelerador de hardware
    → GPU
```

Depois execute a célula abaixo.

In [1]:
import platform
import sys

import torch

print("Python:", sys.version.split()[0])
print("Sistema:", platform.platform())
print("PyTorch:", torch.__version__)
print("CUDA disponível:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    raise RuntimeError(
        "GPU não está disponível. "
        "No Colab, ative uma GPU antes de continuar."
    )

Python: 3.13.15
Sistema: Linux-6.6.122+-x86_64-with-glibc2.39
PyTorch: 2.11.0+cu128
CUDA disponível: True
GPU: Tesla T4


## 2. Instalar Ultralytics

A versão instalada será registrada para manter a rastreabilidade do experimento.

In [2]:
!pip install -q ultralytics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.6/46.6 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 38.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.8/75.8 kB 8.8 MB/s eta 0:00:00


In [3]:
import ultralytics

from ultralytics import YOLO

print("Ultralytics:", ultralytics.__version__)

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
Ultralytics: 8.4.150


## 3. Montar o Google Drive

O Drive será utilizado para:

- ler o ZIP do dataset;
- salvar pesos do modelo;
- salvar métricas;
- salvar gráficos;
- preservar os resultados mesmo após o encerramento da sessão do Colab.

In [4]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


## 4. Configurar caminhos e parâmetros do experimento

Para o primeiro baseline usaremos:

- modelo: `yolov8n.pt`;
- resolução: `640 x 640`;
- épocas: `50`;
- batch automático;
- seed: `42`.

Esses valores poderão ser alterados em experimentos futuros, mas o primeiro baseline deve permanecer registrado.

In [5]:
from pathlib import Path

# -----------------------------
# Caminhos no Google Drive
# -----------------------------
DRIVE_TCC = Path("/content/drive/MyDrive/TCC_Visao_Computacional")

ZIP_DATASET = DRIVE_TCC / "datasets" / "homeobjects_yolo.zip"
PASTA_RESULTADOS_DRIVE = DRIVE_TCC / "resultados" / "yolov8n_baseline"

# -----------------------------
# Caminhos temporários do Colab
# -----------------------------
PASTA_TRABALHO = Path("/content/tcc_visao_computacional")
PASTA_DATASET = PASTA_TRABALHO / "homeobjects_yolo"

# -----------------------------
# Parâmetros do experimento
# -----------------------------
MODELO_BASE = "yolov8n.pt"
EPOCHS = 50
IMGSZ = 640
BATCH = -1
SEED = 42
PATIENCE = 15
WORKERS = 2

# Treinamento normal
MODO_TESTE_RAPIDO = False

if MODO_TESTE_RAPIDO:
    EPOCHS = 3

PASTA_RESULTADOS_DRIVE.mkdir(parents=True, exist_ok=True)

print("ZIP do dataset:", ZIP_DATASET)
print("Resultados:", PASTA_RESULTADOS_DRIVE)
print()
print("Modelo:", MODELO_BASE)
print("Épocas:", EPOCHS)
print("Imagem:", IMGSZ)
print("Batch:", BATCH)
print("Seed:", SEED)

ZIP do dataset: /content/drive/MyDrive/TCC_Visao_Computacional/datasets/homeobjects_yolo.zip
Resultados: /content/drive/MyDrive/TCC_Visao_Computacional/resultados/yolov8n_baseline

Modelo: yolov8n.pt
Épocas: 50
Imagem: 640
Batch: -1
Seed: 42


## 5. Verificar se o dataset está no Drive

Se esta célula gerar erro, confira se o arquivo está exatamente em:

```text
MyDrive/TCC_Visao_Computacional/datasets/homeobjects_yolo.zip
```

In [6]:
if not ZIP_DATASET.exists():
    raise FileNotFoundError(
        f"Dataset não encontrado em:\n{ZIP_DATASET}\n\n"
        'Arquivo homeobjects_yolo.zip esperado na pasta indicada no Google Drive.'
    )

tamanho_mb = ZIP_DATASET.stat().st_size / (1024 ** 2)

print("Dataset encontrado.")
print(f"Tamanho do ZIP: {tamanho_mb:.2f} MB")

Dataset encontrado.
Tamanho do ZIP: 389.41 MB


## 6. Extrair o dataset no armazenamento temporário do Colab

O treinamento ocorrerá utilizando o armazenamento local do Colab, que é mais rápido do que ler milhares de imagens diretamente do Google Drive.

In [7]:
import shutil

if PASTA_TRABALHO.exists():
    shutil.rmtree(PASTA_TRABALHO)

PASTA_TRABALHO.mkdir(parents=True, exist_ok=True)

shutil.unpack_archive(
    filename=str(ZIP_DATASET),
    extract_dir=str(PASTA_TRABALHO),
)

print("ZIP extraído.")

ZIP extraído.


## 7. Localizar automaticamente o `data.yaml`

Essa etapa torna o notebook tolerante ao caso do ZIP conter ou não uma pasta adicional na raiz.

In [8]:
arquivos_yaml = list(PASTA_TRABALHO.rglob("data.yaml"))

if len(arquivos_yaml) == 0:
    raise FileNotFoundError("Nenhum data.yaml foi encontrado após a extração.")

if len(arquivos_yaml) > 1:
    print("Foram encontrados vários data.yaml:")
    for arquivo in arquivos_yaml:
        print(" -", arquivo)

    raise RuntimeError(
        "Há mais de um data.yaml. "
        "Mantenha apenas o dataset homeobjects_yolo no ZIP."
    )

CAMINHO_YAML_ORIGINAL = arquivos_yaml[0]
RAIZ_DATASET = CAMINHO_YAML_ORIGINAL.parent

print("Dataset localizado em:", RAIZ_DATASET)
print("YAML original:", CAMINHO_YAML_ORIGINAL)

Dataset localizado em: /content/tcc_visao_computacional/homeobjects_yolo
YAML original: /content/tcc_visao_computacional/homeobjects_yolo/data.yaml


## 8. Ajustar o `data.yaml` para o ambiente Colab

O YAML criado no Windows contém um caminho absoluto do computador local.

Aqui criamos uma cópia específica para o Colab, mantendo:

- `train`;
- `val`;
- `test`;
- as 12 classes.

Apenas o campo `path` é atualizado.

In [9]:
import yaml

with CAMINHO_YAML_ORIGINAL.open("r", encoding="utf-8") as arquivo:
    dados_yaml = yaml.safe_load(arquivo)

dados_yaml["path"] = RAIZ_DATASET.as_posix()

CAMINHO_YAML_COLAB = RAIZ_DATASET / "data_colab.yaml"

with CAMINHO_YAML_COLAB.open("w", encoding="utf-8") as arquivo:
    yaml.safe_dump(
        dados_yaml,
        arquivo,
        allow_unicode=True,
        sort_keys=False,
    )

print(CAMINHO_YAML_COLAB.read_text(encoding="utf-8"))

path: /content/tcc_visao_computacional/homeobjects_yolo
train: images/train
val: images/val
test: images/test
names:
  0: bed
  1: sofa
  2: chair
  3: table
  4: lamp
  5: tv
  6: laptop
  7: wardrobe
  8: window
  9: door
  10: potted plant
  11: photo frame



## 9. Validar a estrutura do dataset

Antes do treinamento, confirmamos novamente a quantidade de imagens e labels.

O resultado esperado da preparação anterior é aproximadamente:

```text
train: 1822
val:    456
test:   403
```

In [10]:
EXTENSOES_IMAGEM = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}


def contar_imagens(pasta: Path) -> int:
    return sum(
        1
        for arquivo in pasta.rglob("*")
        if arquivo.is_file()
        and arquivo.suffix.lower() in EXTENSOES_IMAGEM
    )


resumo_dataset = []

for split in ("train", "val", "test"):
    pasta_imagens = RAIZ_DATASET / "images" / split
    pasta_labels = RAIZ_DATASET / "labels" / split

    qtd_imagens = contar_imagens(pasta_imagens)
    qtd_labels = len(list(pasta_labels.rglob("*.txt")))

    resumo_dataset.append(
        {
            "split": split,
            "imagens": qtd_imagens,
            "labels": qtd_labels,
        }
    )

import pandas as pd

df_resumo = pd.DataFrame(resumo_dataset)
display(df_resumo)

if not (df_resumo["imagens"] == df_resumo["labels"]).all():
    raise RuntimeError(
        "A quantidade de imagens e labels não coincide."
    )

if df_resumo["imagens"].sum() == 0:
    raise RuntimeError("Dataset vazio.")

print("Estrutura validada.")

,split,imagens,labels
0,train,1822,1822
1,val,456,456
2,test,403,403


Estrutura validada.


## 10. Registrar informações do experimento

Salvamos os principais parâmetros antes do treinamento para garantir a rastreabilidade metodológica.

In [11]:
import json
from datetime import datetime

informacoes_experimento = {
    "data_execucao_utc": datetime.utcnow().isoformat(),
    "modelo_base": MODELO_BASE,
    "epochs": EPOCHS,
    "imgsz": IMGSZ,
    "batch": BATCH,
    "seed": SEED,
    "patience": PATIENCE,
    "workers": WORKERS,
    "pytorch": torch.__version__,
    "ultralytics": ultralytics.__version__,
    "gpu": torch.cuda.get_device_name(0),
    "dataset": df_resumo.to_dict(orient="records"),
    "classes": dados_yaml["names"],
}

caminho_info = PASTA_RESULTADOS_DRIVE / "configuracao_experimento.json"

with caminho_info.open("w", encoding="utf-8") as arquivo:
    json.dump(
        informacoes_experimento,
        arquivo,
        ensure_ascii=False,
        indent=2,
    )

print(json.dumps(informacoes_experimento, ensure_ascii=False, indent=2))

/tmp/ipykernel_1662/3357213205.py:5: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "data_execucao_utc": datetime.utcnow().isoformat(),


{
  "data_execucao_utc": "2026-09-13T16:31:38.569244",
  "modelo_base": "yolov8n.pt",
  "epochs": 50,
  "imgsz": 640,
  "batch": -1,
  "seed": 42,
  "patience": 15,
  "workers": 2,
  "pytorch": "2.11.0+cu128",
  "ultralytics": "8.4.150",
  "gpu": "Tesla T4",
  "dataset": [
    {
      "split": "train",
      "imagens": 1822,
      "labels": 1822
    },
    {
      "split": "val",
      "imagens": 456,
      "labels": 456
    },
    {
      "split": "test",
      "imagens": 403,
      "labels": 403
    }
  ],
  "classes": {
    "0": "bed",
    "1": "sofa",
    "2": "chair",
    "3": "table",
    "4": "lamp",
    "5": "tv",
    "6": "laptop",
    "7": "wardrobe",
    "8": "window",
    "9": "door",
    "10": "potted plant",
    "11": "photo frame"
  }
}


# 11. Treinamento baseline

Agora iniciamos o fine-tuning do **YOLOv8n pré-treinado**.

O modelo utilizará somente:

```text
train
   +
validation
```

O conjunto `test` continuará reservado.

In [12]:
modelo = YOLO(MODELO_BASE)

resultado_treino = modelo.train(
    data=str(CAMINHO_YAML_COLAB),
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    patience=PATIENCE,
    seed=SEED,
    deterministic=True,
    workers=WORKERS,
    device=0,
    pretrained=True,
    optimizer="auto",
    plots=True,
    save=True,
    verbose=True,
    project="/content/experimentos",
    name="yolov8n_baseline",
    exist_ok=True,
)

Ultralytics 8.4.150 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=-1, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/tcc_visao_computacional/homeobjects_yolo/data_colab.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolov8n_bas

## 12. Localizar o melhor modelo

O Ultralytics salva dois pesos principais:

```text
best.pt
last.pt
```

Para as próximas etapas do TCC utilizaremos principalmente o **`best.pt`**.

In [13]:
PASTA_RUN = Path("/content/experimentos/yolov8n_baseline")
CAMINHO_BEST = PASTA_RUN / "weights" / "best.pt"
CAMINHO_LAST = PASTA_RUN / "weights" / "last.pt"

if not CAMINHO_BEST.exists():
    raise FileNotFoundError(
        f"O best.pt não foi encontrado em {CAMINHO_BEST}"
    )

print("Best:", CAMINHO_BEST)
print("Last:", CAMINHO_LAST)
print()
print(f"Tamanho best.pt: {CAMINHO_BEST.stat().st_size / 1024**2:.2f} MB")

Best: /content/experimentos/yolov8n_baseline/weights/best.pt
Last: /content/experimentos/yolov8n_baseline/weights/last.pt

Tamanho best.pt: 5.95 MB


# 13. Avaliação final no conjunto TEST

Esta é uma etapa importante para o TCC.

O modelo treinado é carregado a partir do `best.pt` e avaliado nas **403 imagens reservadas para teste**.

As principais métricas de interesse serão:

- Precision;
- Recall;
- mAP@50;
- mAP@50-95.

Além disso, o Ultralytics gerará gráficos e matriz de confusão.

In [16]:
melhor_modelo = YOLO(str(CAMINHO_BEST))

resultado_test = melhor_modelo.val(
    data=str(CAMINHO_YAML_COLAB),
    split="test",
    imgsz=IMGSZ,
    batch=16,
    device=0,
    plots=True,
    verbose=True,
    project="/content/experimentos",
    name="yolov8n_baseline_test",
    exist_ok=True,
)

Ultralytics 8.4.150 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 72 layers, 3,007,988 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 130.9±68.8 MB/s, size: 163.3 KB)
val: Scanning /content/tcc_visao_computacional/homeobjects_yolo/labels/test.cache... 403 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 403/403 105.6Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 26/26 2.8it/s 9.1s
                   all        403       3465      0.642       0.65      0.652      0.446
                   bed         22         22      0.654      0.682      0.716      0.457
                  sofa        285        397      0.816      0.806      0.855      0.642
                 chair        154        305      0.668      0.721      0.683      0.469
                 table        299        467      0.705      0.745      0.795      0.564
                

## 14. Resumo das métricas do TEST

As métricas abaixo serão salvas também em CSV para uso posterior nos capítulos do TCC.

In [17]:
metricas_test = {
    "precision": float(resultado_test.box.mp),
    "recall": float(resultado_test.box.mr),
    "map50": float(resultado_test.box.map50),
    "map50_95": float(resultado_test.box.map),
}

df_metricas_test = pd.DataFrame(
    [
        {"metrica": metrica, "valor": valor}
        for metrica, valor in metricas_test.items()
    ]
)

display(df_metricas_test)

df_metricas_test.to_csv(
    PASTA_RESULTADOS_DRIVE / "metricas_test.csv",
    index=False,
    encoding="utf-8-sig",
)

,metrica,valor
0,precision,0.642256
1,recall,0.650395
2,map50,0.652211
3,map50_95,0.446108


## 15. Métricas por classe

Como o HomeObjects-3K possui forte desbalanceamento entre classes, não devemos avaliar o modelo apenas pelo mAP global.

Esta etapa registra o desempenho individual das 12 classes.

In [18]:
nomes_classes = melhor_modelo.names

metricas_classes = []

for indice, classe_id in enumerate(resultado_test.box.ap_class_index):
    classe_id = int(classe_id)

    metricas_classes.append(
        {
            "id_classe": classe_id,
            "classe": nomes_classes[classe_id],
            "precision": float(resultado_test.box.p[indice]),
            "recall": float(resultado_test.box.r[indice]),
            "map50": float(resultado_test.box.ap50[indice]),
            "map50_95": float(resultado_test.box.ap[indice]),
        }
    )

df_metricas_classes = pd.DataFrame(metricas_classes)

display(
    df_metricas_classes.sort_values(
        "map50_95",
        ascending=False,
    )
)

df_metricas_classes.to_csv(
    PASTA_RESULTADOS_DRIVE / "metricas_por_classe_test.csv",
    index=False,
    encoding="utf-8-sig",
)

,id_classe,classe,precision,recall,map50,map50_95
1,1,sofa,0.816242,0.806045,0.855212,0.642411
11,11,photo frame,0.821081,0.753926,0.813327,0.639351
5,5,tv,0.684973,0.740741,0.752095,0.612176
3,3,table,0.704871,0.745182,0.795277,0.563938
2,2,chair,0.668419,0.721311,0.682573,0.469446
0,0,bed,0.654335,0.681818,0.715778,0.457307
7,7,wardrobe,0.639251,0.587156,0.606391,0.402310
10,10,potted plant,0.723022,0.679797,0.702108,0.375446
8,8,window,0.539549,0.579515,0.528921,0.327328
9,9,door,0.407118,0.492803,0.442125,0.310443


## 16. Copiar todos os resultados para o Google Drive

Serão preservados:

- `best.pt`;
- `last.pt`;
- `results.csv`;
- curvas de treinamento;
- matriz de confusão;
- gráficos de Precision/Recall;
- avaliação do TEST;
- configurações do experimento.

In [19]:
import shutil

DESTINO_TREINO = PASTA_RESULTADOS_DRIVE / "treinamento"
DESTINO_TEST = PASTA_RESULTADOS_DRIVE / "avaliacao_test"

if DESTINO_TREINO.exists():
    shutil.rmtree(DESTINO_TREINO)

if DESTINO_TEST.exists():
    shutil.rmtree(DESTINO_TEST)

shutil.copytree(
    "/content/experimentos/yolov8n_baseline",
    DESTINO_TREINO,
)

shutil.copytree(
    "/content/experimentos/yolov8n_baseline_test",
    DESTINO_TEST,
)

# Cópia direta do principal artefato do projeto.
shutil.copy2(
    CAMINHO_BEST,
    PASTA_RESULTADOS_DRIVE / "best.pt",
)

print("Resultados salvos em:")
print(PASTA_RESULTADOS_DRIVE)

Resultados salvos em:
/content/drive/MyDrive/TCC_Visao_Computacional/resultados/yolov8n_baseline


## 17. Testar o modelo em algumas imagens do TEST

Esta etapa gera exemplos visuais das predições do modelo treinado.

Ela **não altera as métricas**. Serve apenas para inspeção qualitativa.

In [20]:
import random

random.seed(SEED)

pasta_test = RAIZ_DATASET / "images" / "test"

imagens_test = [
    arquivo
    for arquivo in pasta_test.rglob("*")
    if arquivo.is_file()
    and arquivo.suffix.lower() in EXTENSOES_IMAGEM
]

quantidade_amostras = min(8, len(imagens_test))

amostras = random.sample(
    imagens_test,
    quantidade_amostras,
)

resultados_predicao = melhor_modelo.predict(
    source=[str(imagem) for imagem in amostras],
    imgsz=IMGSZ,
    conf=0.25,
    device=0,
    save=True,
    project="/content/experimentos",
    name="amostras_test",
    exist_ok=True,
)

print(f"{quantidade_amostras} imagens processadas.")


0: 640x640 3 sofas, 4 chairs, 4 tables, 1 lamp, 2 windows, 1 door, 3 potted plants, 4.9ms
1: 640x640 2 sofas, 2 chairs, 1 table, 5 photo frames, 4.9ms
2: 640x640 2 chairs, 1 table, 1 lamp, 2 windows, 2 photo frames, 4.9ms
3: 640x640 1 table, 1 lamp, 1 wardrobe, 4 windows, 5 potted plants, 4.9ms
4: 640x640 1 sofa, 2 chairs, 4 tables, 4 lamps, 3 windows, 3 potted plants, 4 photo frames, 4.9ms
5: 640x640 1 sofa, 1 lamp, 1 tv, 2 doors, 1 potted plant, 4.9ms
6: 640x640 3 sofas, 6 chairs, 9 tables, 3 lamps, 3 potted plants, 4.9ms
7: 640x640 1 chair, 1 wardrobe, 4 potted plants, 4.9ms
Speed: 2.1ms preprocess, 4.9ms inference, 0.8ms postprocess per image at shape (1, 3, 640, 640)
Results saved to /content/experimentos/amostras_test
8 imagens processadas.


## 18. Salvar exemplos de predição no Drive

In [21]:
DESTINO_AMOSTRAS = PASTA_RESULTADOS_DRIVE / "amostras_test"

if DESTINO_AMOSTRAS.exists():
    shutil.rmtree(DESTINO_AMOSTRAS)

shutil.copytree(
    "/content/experimentos/amostras_test",
    DESTINO_AMOSTRAS,
)

print("Amostras salvas em:")
print(DESTINO_AMOSTRAS)

Amostras salvas em:
/content/drive/MyDrive/TCC_Visao_Computacional/resultados/yolov8n_baseline/amostras_test


# 19. Resultado esperado

Após a execução completa, seu Google Drive deverá conter:

```text
TCC_Visao_Computacional/
│
├── datasets/
│   └── homeobjects_yolo.zip
│
└── resultados/
    └── yolov8n_baseline/
        │
        ├── best.pt
        ├── configuracao_experimento.json
        ├── metricas_test.csv
        ├── metricas_por_classe_test.csv
        │
        ├── treinamento/
        │   ├── weights/
        │   │   ├── best.pt
        │   │   └── last.pt
        │   ├── results.csv
        │   └── ...
        │
        ├── avaliacao_test/
        │   ├── confusion_matrix.png
        │   └── ...
        │
        └── amostras_test/
```

## Próxima etapa

Depois deste baseline, teremos dados suficientes para analisar:

1. se o YOLOv8n aprendeu adequadamente as 12 classes;
2. quais classes apresentam maior dificuldade;
3. se o desbalanceamento afetou `bed`, `laptop`, `door` etc.;
4. se devemos ajustar hiperparâmetros ou manter o baseline;
5. iniciar o módulo de **avaliação de risco à locomoção**.

**Não altere o conjunto TEST entre experimentos.**